# 第 07 章 · 运行配置：身份、模型与生效快照

一次请求开始后，配置可能更新、账号可能切换、模型目录刷新也可能晚到。
如果每次读字段都取“当前值”，同一个请求就可能组合出从未存在过的配置、身份和能力。
解决办法是先发布一致的状态，再捕获不可变请求快照；异步返回结果安装前重新核对身份。

本章使用虚构主体与内存模型目录，只验证版本和所有权规则。`AuthLease` 是身份版本值，
没有 token；任何变量都不代表真实凭据，不读取配置文件或环境变量。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
snapshots = load_example('07', 'snapshots')
from dataclasses import replace
models = (
    snapshots.ModelCapabilities('small', frozenset({'text'})),
    snapshots.ModelCapabilities('code', frozenset({'text', 'tools'})),
)
def catalog(ticket, version='catalog-1'):
    return snapshots.CatalogSnapshot(version, ticket.config_version,
        ticket.provider, ticket.actor, ticket.source_generation, models)

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. 先验证目录，再捕获一个完整请求


    Config 指定使用什么 provider/model，身份租约说明“为谁”，Catalog 描述对应主体可用的能力。
    只有三者匹配才捕获请求。读取模型目录失败时偷偷挑另一个模型，会改变用户选择和能力合同。
    修改 `required`，观察能力不足时捕获失败，而不是自动换模型。


In [3]:
config = snapshots.ConfigSnapshot('config-1', 'fixture-provider', 'code')
owner = snapshots.SnapshotOwner(config, 'alice')
ticket = owner.begin_refresh()
assert owner.install(ticket, catalog(ticket))
required = frozenset({'text', 'tools'})
request = owner.capture('request-1', required)
assert request.config.model == 'code'
assert request.lease.actor == 'alice'
assert required <= request.capabilities.supports
request


RequestSnapshot(request_id='request-1', config=ConfigSnapshot(version='config-1', provider='fixture-provider', model='code'), lease=AuthLease(actor='alice', source_generation=1, credential_revision=1), catalog_version='catalog-1', capabilities=ModelCapabilities(model='code', supports=frozenset({'tools', 'text'})))

## 2. 配置变化影响下一次捕获，不改写已捕获请求


    请求快照是一个事实：request-1 当时选择了什么。更新全局配置不能追溯改变它。
    但新配置配旧目录也不合法，下一次捕获应等待对应版本的目录发布。
    修改 `next_model` 为 small，随后显式要求 tools，会得到能力错误；原请求继续保持 code。


In [4]:
next_model = 'small'
owner.update_config(snapshots.ConfigSnapshot('config-2', 'fixture-provider', next_model))
mismatch = expect_error(snapshots.RouteError, lambda: owner.capture('request-2', frozenset({'text'})), 'mismatch')
next_ticket = owner.begin_refresh()
assert owner.install(next_ticket, catalog(next_ticket, 'catalog-2'))
next_request = owner.capture('request-2', frozenset({'text'}))
assert request.config.version == 'config-1' and request.config.model == 'code'
assert next_request.config.version == 'config-2' and next_request.config.model == next_model
capability_error = expect_error(snapshots.RouteError, lambda: owner.capture('requires-tools', frozenset({'tools'})), 'unsupported_capability')
{'old_model': request.config.model, 'new_model': next_request.config.model, 'stale_catalog': mismatch, 'capability_error': capability_error}


{'old_model': 'code',
 'new_model': 'small',
 'stale_catalog': 'catalog_identity_or_config_mismatch',
 'capability_error': 'unsupported_capability'}

## 3. 账号切换后，晚到目录不能覆盖当前身份


    刷新开始时发出 ticket，里面绑定配置版本、provider、actor 和 source generation。
    返回的数据不仅要与 ticket 一致，ticket 还要与当前 owner 一致。只比较模型名或缓存 etag 不够。
    即使重新登录同一个名字，来源代次也可能变化，不能把两个会话当成同一身份来源。


In [5]:
alice_ticket = owner.begin_refresh()
owner.switch_actor('bob')
assert not owner.install(alice_ticket, catalog(alice_ticket, 'late-alice'))
expect_error(snapshots.RouteError, lambda: owner.capture('bob-before-refresh', frozenset()), 'mismatch')
bob_ticket = owner.begin_refresh()
assert owner.install(bob_ticket, catalog(bob_ticket, 'bob-catalog'))
bob_request = owner.capture('bob-1', frozenset({'text'}))
assert bob_request.lease.actor == 'bob'
prior_generation = bob_request.lease.source_generation
relogin_ticket = owner.begin_refresh()
owner.switch_actor('bob')
assert not owner.install(relogin_ticket, catalog(relogin_ticket, 'old-bob-session'))
current_ticket = owner.begin_refresh()
assert owner.install(current_ticket, catalog(current_ticket, 'new-bob-session'))
assert owner.capture('bob-2', frozenset()).lease.source_generation > prior_generation
{'old_actor_preserved_in_request': request.lease.actor, 'current_actor': owner.capture('current', frozenset()).lease.actor}


{'old_actor_preserved_in_request': 'alice', 'current_actor': 'bob'}

## 4. 取消刷新保留上一份成功目录


    刷新取消不应把可用快照先清空，也不允许晚到结果绕过取消重新发布。
    ticket 被 owner 移除后，即使结果内容完全匹配，也只能被视为已失效工作。
    本例取消的是目录刷新，不声称中断了真实 HTTP 请求。


In [6]:
before_cancel = owner.capture('before-cancel', frozenset({'text'}))
cancelled_ticket = owner.begin_refresh()
owner.cancel_refresh(cancelled_ticket)
installed = owner.install(cancelled_ticket, catalog(cancelled_ticket, 'should-not-install'))
after_cancel = owner.capture('after-cancel', frozenset({'text'}))
assert installed is False
assert after_cancel.catalog_version == before_cancel.catalog_version
assert after_cancel.lease == before_cancel.lease
{'late_install_accepted': installed, 'retained_catalog': after_cancel.catalog_version}


{'late_install_accepted': False, 'retained_catalog': 'new-bob-session'}

## 5. 同来源续期与身份替换是不同操作


    同主体、同来源的凭据续期只增加 credential revision；换账号或重新登录增加 source generation。
    这个区分允许下一次请求使用新凭据，同时保留旧请求的身份记录。
    续期仍须核对当前 owner，不能把 Alice 的过期请求拿来续 Bob 的身份。


In [7]:
current = owner.capture('before-renew', frozenset())
renewed_lease = owner.renew_same_source(current.lease)
next_capture = owner.capture('after-renew', frozenset())
assert renewed_lease.source_generation == current.lease.source_generation
assert renewed_lease.credential_revision == current.lease.credential_revision + 1
assert next_capture.lease == renewed_lease
assert current.lease != renewed_lease
stale_identity = expect_error(snapshots.RouteError, lambda: owner.renew_same_source(request.lease), 'identity_changed')
{'old_revision': current.lease.credential_revision, 'new_revision': renewed_lease.credential_revision, 'cross_identity_renewal': stale_identity}


{'old_revision': 1,
 'new_revision': 2,
 'cross_identity_renewal': 'identity_changed'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [PROJECT_LOCAL_CONFIG_DENYLIST / sanitize_project_config](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/config/src/loader/mod.rs#L90-L104)：受信项目仍不能指定 provider/base URL 等敏感顶层键；sanitize 删除并收集 ignored_keys。

- [version_for_toml](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/config/src/fingerprint.rs#L54-L65)：内容版本对排序后的 JSON 序列化计算 SHA-256。

- [fetch_and_update_models / apply_remote_models](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/models-manager/src/manager.rs#L905-L930)：目录返回及异步 cache.store 之后发布前都检查当前 identity，晚到旧身份不能覆盖内存目录。


## 现在应能解释什么


    能区分配置版本、身份来源代次、凭据修订和目录版本；能解释为什么已经准备的请求不受热更新追溯修改。
    接回 Loop 时在请求边界捕获一份一致快照，异步目录结果经 owner 验证后发布。

    本章复用纯内存快照原型；没有实现真实登录、token 刷新、配置优先级解析或凭据落盘。
